# ForgeLM — GRPO Reasoning RL

Train a model to reason step-by-step using Group Relative Policy Optimization (the method behind DeepSeek-R1).

**Requirements:** GPU runtime required (Runtime → Change runtime type → T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/grpo_reasoning.ipynb)

In [ ]:
# Step 1: Install ForgeLM (with bitsandbytes for 4-bit quantization)
# Pinned to v0.5.6; bump on each release
!pip install -q --no-cache-dir 'forgelm[qlora]==0.5.6' bitsandbytes
!pip uninstall -y wandb -q
!forgelm --version

In [ ]:
# Step 2: Check GPU
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)
    print(f"GPU: {gpu_name} ({vram_gb} GB VRAM)")
else:
    print("WARNING: No GPU. GRPO requires GPU. Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Step 3: Create GRPO dataset (prompt-only — model generates responses during training)
import json

prompts = [
    {"prompt": "What is 15% of 240? Think step by step."},
    {"prompt": "If a train travels 60 km/h for 2.5 hours, how far does it go?"},
    {"prompt": "What is the sum of the first 10 prime numbers?"},
    {"prompt": "A rectangle has width 5 and perimeter 22. What is the length?"},
    {"prompt": "If 3x + 7 = 22, what is x?"},
    {"prompt": "What is 2^10?"},
    {"prompt": "A store offers 20% off. Original price is $80. What's the final price?"},
    {"prompt": "How many seconds are in 3 hours?"},
]

with open("math_prompts.jsonl", "w") as f:
    for p in prompts:
        f.write(json.dumps(p) + "\n")

print(f"Created {len(prompts)} GRPO math prompts")

In [ ]:
# Step 4: Create config
config_yaml = """
model:
  name_or_path: "HuggingFaceTB/SmolLM2-135M-Instruct"
  max_length: 512
  load_in_4bit: true

lora:
  r: 16
  alpha: 32
  target_modules: ["q_proj", "v_proj"]

training:
  trainer_type: "grpo"
  grpo_num_generations: 4
  grpo_max_new_tokens: 256
  output_dir: "./grpo_checkpoints"
  max_steps: 50
  per_device_train_batch_size: 1
  learning_rate: 1.0e-6

data:
  dataset_name_or_path: "math_prompts.jsonl"
"""

with open("grpo_config.yaml", "w") as f:
    f.write(config_yaml)
print("GRPO config saved! (max_steps: 50)")

In [ ]:
# Step 5: Validate
!forgelm --config grpo_config.yaml --dry-run

In [ ]:
# Step 6: Train
!forgelm --config grpo_config.yaml

In [ ]:
# Step 7: Verify output
import os

model_path = "./grpo_checkpoints/final_model"
if not os.path.exists(model_path) or not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print("ERROR: Model not found. Check training output above.")
else:
    print(f"Training completed! Model saved to: {model_path}")
    print(f"Files: {os.listdir(model_path)}")

In [ ]:
# Step 8: Compare base model vs GRPO-trained model
import os

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "./grpo_checkpoints/final_model"
base_model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

if not os.path.exists(model_path) or not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print("Error: Model not found. Ensure training completed successfully.")
else:
    print("Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    print("Loading GRPO-trained model...")
    ft_model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(base_model_name), model_path)
    ft_tokenizer = AutoTokenizer.from_pretrained(model_path)

    # Use UNSEEN math problems (not in training data) to test generalization
    test_prompts = [
        "What is 25% of 160? Think step by step.",
        "A car drives 90 km in 1.5 hours. What is its speed?",
        "If 2x + 5 = 15, what is x? Show your work.",
    ]

    for prompt in test_prompts:
        messages = [{"role": "user", "content": prompt}]
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors="pt")

        print(f"\n{'=' * 60}")
        print(f"PROMPT: {prompt}")
        print(f"{'=' * 60}")

        base_out = base_model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
        base_resp = tokenizer.decode(base_out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()
        print(f"\n[BASE MODEL]:\n{base_resp[:400]}")

        ft_inputs = ft_tokenizer(formatted, return_tensors="pt")
        ft_out = ft_model.generate(**ft_inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
        ft_resp = ft_tokenizer.decode(ft_out[0][ft_inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()
        print(f"\n[GRPO-TRAINED]:\n{ft_resp[:400]}")

    print(f"\n{'=' * 60}")
    print("GRPO-trained model should show more structured, step-by-step reasoning.")

## How GRPO Works

1. For each prompt, the model generates `num_generations` responses
2. Responses are scored by a reward function (correctness for math)
3. The model is updated to prefer higher-scoring responses
4. No human preference data needed — the model learns from its own outputs

This is the method behind DeepSeek-R1's reasoning capabilities.